# 10 -- A Learned CrowdRL Policy as a JuPedSim Operational Model

This notebook recreates the flow of JuPedSim's
[getting started guide](https://www.jupedsim.org/stable/notebooks/getting_started.html),
with one substitution: instead of a built-in operational model (e.g. the
Collision Free Speed Model), the agents are driven by a **trained CrowdRL
policy** deployed through `crowdrl_jupedsim.LearnedPolicyModel`.

The policy artefact is the repo's `example_model/policy_r0400.onnx` -- a
**self-describing** ONNX (issue #7): its resolved observation/action configs
and provenance are embedded in the file's metadata, so the adapter needs no
hand-supplied configuration at all.

Scenarios (the same two the e2e test `tests/test_e2e_jupedsim_trained_policy.py`
pins):

1. **Corner** -- the L-corridor from
   [jupedsim#1625](https://github.com/PedestrianDynamics/jupedsim/issues/1625):
   agents must follow the routed waypoint around a bend they cannot see the
   goal through.
2. **Bottleneck** -- 12 agents through a 1.4 m aperture: crowd behaviour,
   not just solo navigation.

Requirements: a JuPedSim 2.0 source build on `sys.path` (the custom-model
layer is not on PyPI -- see `plan/handover_2026-07-29.md` for the local
wiring) plus the `dev` extra (`pedpy`, `plotly`) for the animation.

In [1]:
from pathlib import Path
import sys

# JuPedSim 2.0 comes from a local source build. If a plain import fails,
# fall back to the known local build paths (machine-specific wiring).
try:
    import jupedsim as jps
except ImportError:
    _JPS = Path("/home/fabi/dev/jupedsim")
    sys.path[:0] = [
        str(_JPS / "python_modules" / "jupedsim"),
        str(_JPS / "build-py312" / "lib"),
    ]
    import jupedsim as jps

import shapely
import plotly.io as pio
from jupedsim.internal.notebook_utils import animate, read_sqlite_file

from crowdrl_jupedsim import CrowdRLAgentState, LearnedPolicyModel, OnnxPolicy

# notebook_utils pins the renderer for the jupedsim.org docs build; the plain
# mimetype renderer keeps outputs small and renders in VS Code / JupyterLab.
pio.renderers.default = "plotly_mimetype"

print("jupedsim", jps.__version__)

jupedsim 2.0.0


## The self-describing policy

`OnnxPolicy` reads the training configuration back out of the artefact.
Nothing below is typed in by hand -- and if we *did* pass an explicit config
that disagrees with the embedded record, construction would raise.

In [2]:
POLICY_PATH = Path("..") / "example_model" / "policy_r0400.onnx"

policy = OnnxPolicy(POLICY_PATH)
meta = policy.metadata
print("obs_dim:", meta.obs_dim, "| action_dim:", meta.action_dim)
print("provenance:", meta.provenance)
print(
    "use_goal_direction =", meta.obs_config.use_goal_direction,
    "-> the policy navigates by the routed next waypoint alone",
)

obs_dim: 89 | action_dim: 4
provenance: {'run': 'results_exp_nogoaldir_stable_bigrooms_density_v4', 'checkpoint': 'checkpoint_rollout_0400.pt', 'rollout': 31398, 'git_rev': '701a41d', 'source': 'scripts/reexport_onnx.py'}
use_goal_direction = False -> the policy navigates by the routed next waypoint alone


In [3]:
OUT_DIR = Path("outputs/10_jupedsim")
OUT_DIR.mkdir(parents=True, exist_ok=True)


def run_scenario(name, area, exit_polygon, spawns, max_steps=6000):
    """Run the learned model in JuPedSim, recording trajectories to sqlite.

    Mirrors the getting-started loop: build the simulation with a
    SqliteTrajectoryWriter (records every 4th frame -> 25 fps at dt=0.01),
    add an exit stage + journey, spawn agents, iterate until everyone left.
    """
    trajectory_file = OUT_DIR / f"{name}.sqlite"
    trajectory_file.unlink(missing_ok=True)

    model = LearnedPolicyModel(OnnxPolicy(POLICY_PATH))  # self-configured
    simulation = jps.Simulation(
        model=model,
        geometry=area,
        dt=0.01,
        trajectory_writer=jps.SqliteTrajectoryWriter(output_file=trajectory_file),
    )
    exit_id = simulation.add_exit_stage(exit_polygon)
    journey_id = simulation.add_journey(jps.JourneyDescription([exit_id]))
    for position in spawns:
        simulation.add_agent(
            journey_id=journey_id,
            stage_id=exit_id,
            state=CrowdRLAgentState(position=position),
        )

    n = simulation.agent_count()
    while simulation.agent_count() > 0 and simulation.iteration_count() < max_steps:
        simulation.iterate()

    print(
        f"{name}: {n - simulation.agent_count()}/{n} agents exited in "
        f"{simulation.iteration_count() * 0.01:.1f} s simulated time"
    )
    return trajectory_file

## Scenario 1 -- the corner (jupedsim#1625 geometry)

An L-shaped corridor: the exit is around a bend, so the straight line to the
goal points through a wall. Before JuPedSim exposed the routed waypoint
(`ped.next_target`, merged in
[PR #1626](https://github.com/PedestrianDynamics/jupedsim/pull/1626)), a
Python model only saw the final goal here -- every agent walked into the
corner wall and pinned. Now the policy receives the waypoint through the
navmesh observation block and rounds the bend.

In [4]:
CORNER_AREA = shapely.Polygon(
    [(0, 0), (12, 0), (12, 12), (10, 12), (10, 2), (0, 2)]
)
CORNER_EXIT = [(10, 11), (12, 11), (12, 12), (10, 12)]
CORNER_SPAWNS = [(1.5, 1.0), (3.0, 1.2), (4.5, 0.8), (6.0, 1.0)]

corner_file = run_scenario("corner", CORNER_AREA, CORNER_EXIT, CORNER_SPAWNS, max_steps=4000)
trajectory_data, walkable_area = read_sqlite_file(corner_file)
animate(trajectory_data, walkable_area, every_nth_frame=5, radius=0.225)

corner: 4/4 agents exited in 12.1 s simulated time


## Scenario 2 -- bottleneck (12 agents, 1.4 m aperture)

An hourglass room pinched to a 1.4 m opening -- mid-range of the tier-1
training distribution. This probes crowd behaviour rather than solo
navigation: queueing, merging and flow through the neck.

Observed on this scenario (see `plan/handover_2026-07-29.md`): all 12 agents
exit in ~12 s of simulated time, a specific flow of roughly 1.2 (m s)^-1 --
near the empirical bottleneck range. **Known caveat:** contact forces are not
wired in the adapter yet (walking-skeleton scope), so agents can momentarily
interpenetrate in the neck (min pairwise distance ~0.04 m here). Navigation
and throughput translate; spacing discipline awaits the shared
contact-force module.

In [5]:
import numpy as np

BOTTLENECK_AREA = shapely.Polygon(
    [
        (0, 0), (6.8, 0), (6.8, 4.3), (7.2, 4.3), (7.2, 0), (14, 0),
        (14, 10), (7.2, 10), (7.2, 5.7), (6.8, 5.7), (6.8, 10), (0, 10),
    ]
)
BOTTLENECK_EXIT = [(13.0, 4.0), (14.0, 4.0), (14.0, 6.0), (13.0, 6.0)]

rng = np.random.default_rng(7)
BOTTLENECK_SPAWNS = [
    (float(x), float(y))
    for x, y in zip(rng.uniform(1.0, 5.5, 12), rng.uniform(1.5, 8.5, 12))
]

bottleneck_file = run_scenario(
    "bottleneck", BOTTLENECK_AREA, BOTTLENECK_EXIT, BOTTLENECK_SPAWNS
)
trajectory_data, walkable_area = read_sqlite_file(bottleneck_file)
animate(trajectory_data, walkable_area, every_nth_frame=5, radius=0.225)

bottleneck: 12/12 agents exited in 12.0 s simulated time


## Notes

- The whole deployment surface used here is two calls:
  `LearnedPolicyModel(OnnxPolicy(path))` and the standard JuPedSim
  simulation loop. Everything else (observation assembly, raycasts,
  temporal memory, action interpretation, integration) lives inside the
  operational model and reuses the exact crowdrl-core code from training.
- The same scenarios are pinned as tests in
  `tests/test_e2e_jupedsim_trained_policy.py`; run them with the jupedsim
  build on `PYTHONPATH`.
- Colour in the animation encodes each agent's current speed (pedpy's
  individual-speed computation, as in the JuPedSim docs).